# Final Guided KeyBERT Notebook

Guided Keyphrase Extraction for Stakeholder Value Identification in Transportation Resilience Literature.

Pipeline:
1. Data generation
2. Preprocessing
3. KeyBERT modeling
4. Category mapping
5. Evaluation setup
6. Visualization
7. Robustness checks


In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)

CURRENT = Path.cwd()
PROJECT_ROOT = CURRENT.parent if CURRENT.name.lower() == "notebooks" else CURRENT

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "papers_metadata.csv"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("CSV exists:", DATA_PATH.exists())


In [ ]:
df = pd.read_csv(DATA_PATH, encoding="cp1252")

if "text" in df.columns and "full_text" not in df.columns:
    df = df.rename(columns={"text": "full_text"})

required_cols = ["paper_id", "title", "full_text"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if "keywords" not in df.columns:
    df["keywords"] = ""

for col in ["title", "full_text", "keywords"]:
    df[col] = df[col].fillna("").astype(str)

df["combined_text"] = df["title"] + " " + df["keywords"] + " " + df["full_text"]

print(df.shape)
print(df.columns.tolist())
df.head()


In [ ]:
def clean_text(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = text.replace("", "'").replace("’", "'").replace("“", '"').replace("”", '"')
    text = text.replace("\x02", " ").replace("\xa0", " ")
    text = text.lower()

    text = re.sub(r"http\S+|www\S+", " ", text)

    # citation and publication noise
    for pattern in [r"\bet al\b", r"\bet\b", r"\bal\b", r"\bdoi\b", r"\barxiv\b"]:
        text = re.sub(pattern, " ", text)

    publisher_terms = [
        "elsevier", "springer", "wiley", "emerald", "taylor francis",
        "american society of civil engineers", "asce", "crc press"
    ]
    for term in publisher_terms:
        text = re.sub(rf"\b{re.escape(term)}\b", " ", text)

    # figure/table/page references
    text = re.sub(r"\bfigure\s*\d+\b", " ", text)
    text = re.sub(r"\bfig\s*\d+\b", " ", text)
    text = re.sub(r"\btable\s*\d+\b", " ", text)
    text = re.sub(r"\bpage\s*\d+\b", " ", text)

    # references section
    text = re.sub(r"\breferences\b.*$", " ", text, flags=re.IGNORECASE | re.DOTALL)

    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["combined_text"].apply(clean_text)
print(df["clean_text"].iloc[0][:700])


## Install packages if needed

Uncomment and run the next line if KeyBERT is not installed.


In [ ]:
# !pip install keybert sentence-transformers scikit-learn


In [ ]:
from keybert import KeyBERT

kw_model = KeyBERT(model="all-MiniLM-L6-v2")
print("KeyBERT model loaded.")


In [ ]:
value_framework = {
    "Flexibility": ["flexibility", "flexible", "adaptive capacity", "flexible recovery", "flexible planning"],
    "Safety": ["safety", "public safety", "traffic safety", "emergency safety", "hazard reduction", "risk reduction", "evacuation safety"],
    "Recoverability": ["recoverability", "recovery", "service recovery", "system recovery", "post-disaster recovery"],
    "Reliability": ["reliability", "reliable", "system reliability", "service reliability", "continuity", "service continuity", "operational continuity"],
    "Functionality": ["functionality", "functional performance", "system function", "network function", "service function", "operational function"],
    "Redundancy": ["redundancy", "redundant", "alternative route", "backup system", "system redundancy", "network redundancy", "substitution"],
    "Connectivity": ["connectivity", "network connectivity", "road connectivity", "regional connectivity", "transport connectivity"],
    "Restoration Speed": ["restoration speed", "rapid restoration", "rapid repair", "repair speed", "quick recovery", "rapid response", "response time", "restoration time"],
    "Economic Recovery": ["economic recovery", "business recovery", "regional recovery", "economic restoration", "local economy", "economic activity"],
    "Robustness": ["robustness", "robust", "system robustness", "structural robustness", "network robustness", "hazard resistance"],
    "Operational Performance": ["operational performance", "operation", "operations", "traffic operation", "system performance", "network performance", "travel performance"],
    "Capacity": ["capacity", "system capacity", "road capacity", "traffic capacity", "network capacity", "service capacity"],
    "Accessibility": ["accessibility", "access", "accessible", "road access", "public access", "transportation access", "service access", "transit access"],
    "Mobility": ["mobility", "travel mobility", "transport mobility", "movement", "travel time", "traffic flow", "evacuation mobility"],
    "Equity": ["equity", "equitable", "fairness", "justice", "social equity", "vulnerable populations", "underserved communities", "inclusion", "social vulnerability"],
    "Public Health Support": ["public health", "health support", "healthcare access", "medical access", "emergency medical", "health service", "hospital access"],
    "Economic Continuity": ["economic continuity", "business continuity", "economic resilience", "supply chain continuity", "financial continuity"],
    "Cost-effectiveness": ["cost-effectiveness", "cost effectiveness", "cost efficient", "cost efficiency", "low cost", "life cycle cost", "recovery cost", "investment efficiency"],
    "Resource Availability": ["resource availability", "available resources", "funding availability", "recovery resources", "emergency resources", "material availability", "staff availability"],
    "Environmental Protection": ["environmental protection", "environmental impact", "ecosystem protection", "ecological protection", "natural resource protection", "habitat protection"],
    "Sustainability": ["sustainability", "sustainable", "long-term sustainability", "sustainable recovery", "climate resilience", "climate adaptation"],
    "Pollution Prevention": ["pollution prevention", "pollution control", "air pollution", "emission reduction", "environmental pollution", "contamination prevention"],
    "Resilience": ["resilience", "resilient", "transportation resilience", "infrastructure resilience", "community resilience", "disaster resilience"],
    "Responsiveness": ["responsiveness", "responsive", "response", "emergency response", "rapid response", "response capability", "response time"],
    "Adaptability": ["adaptability", "adaptable", "adaptation", "adaptive", "adaptive planning", "climate adaptation", "adaptive capacity"],
    "Serviceability": ["serviceability", "serviceable", "service performance", "service restoration", "service availability", "transport service", "level of service"]
}

print("Number of value categories:", len(value_framework))


In [ ]:
TOP_N = 20

all_doc_keywords = []

for _, row in df.iterrows():
    keywords = kw_model.extract_keywords(
        row["clean_text"],
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=TOP_N,
        use_mmr=True,
        diversity=0.7
    )

    for phrase, score in keywords:
        all_doc_keywords.append({
            "paper_id": row["paper_id"],
            "title": row["title"],
            "phrase": phrase,
            "score": score
        })

keywords_df = pd.DataFrame(all_doc_keywords)
print("Extracted phrase rows:", keywords_df.shape[0])
keywords_df.head(20)


In [ ]:
phrase_summary = (
    keywords_df.groupby("phrase")
    .agg(
        doc_count=("paper_id", "nunique"),
        occurrence_count=("paper_id", "count"),
        mean_score=("score", "mean"),
        total_score=("score", "sum")
    )
    .reset_index()
    .sort_values(["doc_count", "mean_score", "total_score"], ascending=False)
)

phrase_summary.head(30)


In [ ]:
def map_phrase_to_categories(phrase, framework):
    phrase_l = phrase.lower()
    matched = []

    for category, refs in framework.items():
        for ref in refs:
            ref_l = ref.lower()
            if ref_l in phrase_l or phrase_l in ref_l:
                matched.append(category)
                break

    return matched if matched else ["Emerging/Unmapped"]

mapped_rows = []

for _, row in phrase_summary.iterrows():
    cats = map_phrase_to_categories(row["phrase"], value_framework)
    for cat in cats:
        mapped_rows.append({
            "phrase": row["phrase"],
            "category": cat,
            "doc_count": row["doc_count"],
            "occurrence_count": row["occurrence_count"],
            "mean_score": row["mean_score"],
            "total_score": row["total_score"]
        })

mapped_df = pd.DataFrame(mapped_rows)
mapped_df.head(20)


In [ ]:
category_summary = (
    mapped_df.groupby("category")
    .agg(
        unique_phrases=("phrase", "nunique"),
        total_doc_support=("doc_count", "sum"),
        avg_phrase_score=("mean_score", "mean")
    )
    .reset_index()
    .sort_values(["total_doc_support", "unique_phrases"], ascending=False)
)

category_summary


In [ ]:
example_rows = []

for category in category_summary["category"]:
    cat_phrases = mapped_df[mapped_df["category"] == category].copy()
    cat_phrases = cat_phrases.sort_values(
        ["doc_count", "mean_score", "total_score"],
        ascending=False
    ).drop_duplicates(subset=["phrase"]).head(5)

    example_rows.append({
        "category": category,
        "example_phrases": "; ".join(cat_phrases["phrase"].tolist())
    })

examples_df = pd.DataFrame(example_rows)
examples_df


## Precision@10 review table

Open the generated CSV and fill `relevant_manual_review` with:
- `1` if the phrase is relevant to the category
- `0` if it is not relevant

Then run the precision calculation cell.


In [ ]:
K = 10

review_rows = []

for category in category_summary["category"]:
    cat_phrases = mapped_df[mapped_df["category"] == category].copy()
    cat_phrases = cat_phrases.sort_values(
        ["doc_count", "mean_score", "total_score"],
        ascending=False
    ).drop_duplicates(subset=["phrase"]).head(K)

    for _, row in cat_phrases.iterrows():
        review_rows.append({
            "category": category,
            "phrase": row["phrase"],
            "doc_count": row["doc_count"],
            "mean_score": row["mean_score"],
            "relevant_manual_review": ""
        })

review_df = pd.DataFrame(review_rows)
review_df.to_csv(OUTPUT_TABLES / "keyphrase_review_table_top10.csv", index=False)
review_df.head(30)


In [ ]:
# After manually filling keyphrase_review_table_top10.csv, uncomment and run:
# reviewed = pd.read_csv(OUTPUT_TABLES / "keyphrase_review_table_top10.csv")
# reviewed["relevant_manual_review"] = pd.to_numeric(reviewed["relevant_manual_review"], errors="coerce").fillna(0)
# precision_df = (
#     reviewed.groupby("category")["relevant_manual_review"]
#     .mean()
#     .reset_index()
#     .rename(columns={"relevant_manual_review": "precision_at_10"})
# )
# precision_df.to_csv(OUTPUT_TABLES / "precision_at_10_by_category.csv", index=False)
# precision_df


In [ ]:
plot_df = category_summary[category_summary["category"] != "Emerging/Unmapped"].copy()
plot_df = plot_df.sort_values("total_doc_support", ascending=False).head(15)

plt.figure(figsize=(11, 6))
plt.bar(plot_df["category"], plot_df["total_doc_support"])
plt.xticks(rotation=45, ha="right")
plt.title("Stakeholder Value Category Support in Literature")
plt.xlabel("Stakeholder Value Category")
plt.ylabel("Total Document Support")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "stakeholder_value_category_support.png", dpi=300)
plt.show()


In [ ]:
emerging_df = mapped_df[mapped_df["category"] == "Emerging/Unmapped"].copy()
emerging_df = emerging_df.sort_values(
    ["doc_count", "mean_score", "total_score"],
    ascending=False
).drop_duplicates(subset=["phrase"])

emerging_top = emerging_df.head(20)

if len(emerging_top) > 0:
    plt.figure(figsize=(10, 7))
    plt.barh(emerging_top["phrase"][::-1], emerging_top["doc_count"][::-1])
    plt.title("Top Emerging / Unmapped Keyphrases")
    plt.xlabel("Document Support")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / "emerging_unmapped_keyphrases.png", dpi=300)
    plt.show()

emerging_top[["phrase", "doc_count", "mean_score"]]


## Robustness checks

This section compares:
1. Top 10 vs top 20 phrase extraction
2. Metadata-only extraction using title + keywords


In [ ]:
def extract_keybert_phrases(dataframe, text_col, top_n=10, ngram_range=(1, 3), diversity=0.7):
    rows = []
    for _, row in dataframe.iterrows():
        keywords = kw_model.extract_keywords(
            row[text_col],
            keyphrase_ngram_range=ngram_range,
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=diversity
        )
        for phrase, score in keywords:
            rows.append({
                "paper_id": row["paper_id"],
                "phrase": phrase,
                "score": score
            })
    return pd.DataFrame(rows)

keywords_top10 = extract_keybert_phrases(df, "clean_text", top_n=10)
df["metadata_text"] = (df["title"] + " " + df["keywords"]).apply(clean_text)
metadata_keywords = extract_keybert_phrases(df, "metadata_text", top_n=10)

print("Top10 extraction rows:", keywords_top10.shape)
print("Metadata-only extraction rows:", metadata_keywords.shape)
metadata_keywords.head()


In [ ]:
keywords_df.to_csv(OUTPUT_TABLES / "all_keybert_keywords.csv", index=False)
phrase_summary.to_csv(OUTPUT_TABLES / "keyphrase_corpus_summary.csv", index=False)
mapped_df.to_csv(OUTPUT_TABLES / "keyphrase_mapped_value_categories.csv", index=False)
category_summary.to_csv(OUTPUT_TABLES / "stakeholder_value_category_summary.csv", index=False)
examples_df.to_csv(OUTPUT_TABLES / "stakeholder_value_example_phrases.csv", index=False)
emerging_top.to_csv(OUTPUT_TABLES / "emerging_unmapped_keyphrases.csv", index=False)
keywords_top10.to_csv(OUTPUT_TABLES / "robustness_top10_keyphrases.csv", index=False)
metadata_keywords.to_csv(OUTPUT_TABLES / "robustness_metadata_only_keyphrases.csv", index=False)

print("Saved all outputs to:", OUTPUT_TABLES)


## Interpretation notes

Use the outputs to answer:

- Which stakeholder values are most represented?
- Which values are weakly represented?
- Which extracted phrases best represent each value?
- Which unmapped phrases suggest emerging concerns?
- Are results stable under robustness checks?

Important limitation: because the stakeholder value framework guides interpretation, describe this as **guided keyphrase extraction**, not unrestricted discovery.
